---

# DSC LOGIKA UI 2026: Credit Risk Prediction

*Predicting 90+ DPD within 12 months of origination for Indonesian consumer loans, scored by the KS statistic.*

---

## IndomaretLabtekV (DSC0052)

- Riantama Putra (Leader)
- Athilla Zaidan Zidna Fann
- Nicholas Wise Saragih Sumbayak

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Exploratory Data Analysis**](#3)
4. [**Feature Engineering**](#4)
5. [**Validation Strategy**](#5)
6. [**Gradient Boosting Models**](#6)
7. [**Tabular Foundation Models**](#7)
8. [**Ensemble and KS Stability**](#8)
9. [**Submission**](#9)


---

# Introduction <a name="1"></a>

---

## Overview

*Credit risk is a central challenge in consumer lending. A lender has to judge, at the moment of the loan decision, whether an application is likely to run into serious repayment difficulty using only the information available at that time.*

*This competition frames the problem on a multi-table Indonesian consumer lending dataset. Each selected loan has a 12-month term, and the goal is to rank applications by their risk of reaching 90 or more days past due within 12 months of origination.*

## Aim

*Build a leak-free scoring pipeline that aggregates six behavioural tables into one row per application and produces a risk score with maximal and stable KS separation on both the public and the private leaderboard.*

## Metric

***The Kolmogorov-Smirnov (KS) statistic** measures the largest vertical gap between the cumulative score distributions of bad loans and good loans. It depends only on the ranking of scores, so calibration does not matter and rank-averaging is a safe way to blend models.*

$$KS = \max_s \left| F_1(s) - F_0(s) \right|$$

## Dataset

*Data sourced from the Kaggle competition DSC LOGIKA UI 2026.*

*The training set contains 70,000 selected applications (13.58% bad) and the test set contains 30,000. Every customer has exactly one selected application, and train and test customers are disjoint. About 30% of customers in both splits are thin-file, with no previous applications, repayment history or account data.*

*The dataset is used to engineer customer-level features that respect the `decision_datetime` information cutoff.*

**Attributes**

1.  `train/test_applications`  : Selected loan (amount, 12-month term, rate tier, installment, purpose)
2.  `customer_profiles`        : Age, region, employment, tenure, declared income, dependents, housing
3.  `previous_applications`    : Historical applications with decision, approved amount and rate
4.  `repayment_history`        : Installment-level amounts, payment dates and days past due
5.  `account_behavior`         : Monthly balances, inflows, outflows and low-balance days
6.  `bureau_observations`      : Up to three bureau reports per application (score, utilization, inquiries)
7.  `ecommerce_transactions`   : Item-level orders with category, payment method and status

**Target**
 `target`       : 1 if the loan reaches 90+ DPD within 12 months, 0 otherwise

## Approach: Behavioural Features + GBDT and Tabular Foundation Model Ensemble

*The approach aggregates every auxiliary table to the customer level with explicit cutoff filters. Window lengths are chosen from the exploratory analysis: repayment lateness is a stable trait, so full-history ratios are used, while e-commerce activity and cash-flow stress ramp up in the 3-4 months before the application, so recent-versus-baseline acceleration features are used.*

*Three gradient boosting libraries (LightGBM, XGBoost on GPU, CatBoost on GPU) are trained with repeated stratified 5-fold CV. Two pretrained tabular foundation models (TabPFN-3.5 on GPU 0 and Causilo on GPU 1) are fitted in-context on a de-duplicated feature subset. Out-of-fold predictions are rank-blended and the blend is checked for KS stability before submission.*

```
7 raw tables
    |
    +-- cutoff-filtered aggregations (one row per application, ~190 features)
         |
    +----+----------+----------+-------------+-----------+
    |    |          |          |             |           |
  LightGBM  XGBoost   CatBoost   TabPFN-3.5    Causilo
    |    |          |          |             |           |
    +----+---- rank blend (weights from OOF KS) ---------+
                     |
              default_probability
```

*Train and test are a random split of the same period (PSI below 0.002 for every feature), so stratified K-fold is the correct validation scheme. The public-versus-private KS gap expected from sampling noise alone is about 0.01, so decisions are made on CV rather than on the public leaderboard.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook targets a Kaggle GPU T4 x2 session with internet enabled (needed once to download the TabPFN-3.5 and Causilo weights). TabPFN weights are gated: accept the licence on Hugging Face and store a token as a Kaggle secret named `TABPFN_TOKEN` (or `HF_TOKEN`). If a foundation model cannot be loaded, its step is skipped and the notebook still completes with the gradient boosting ensemble. Estimated runtime is about 1-2 hours on the following GPU.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q polars==2.0.0 lightgbm xgboost catboost tabpfn
%pip install -q --no-deps causilo

## Import Libraries

In [ ]:
import gc
import os
import random
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from scipy.stats import ks_2samp, rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

try:
    from tabpfn import TabPFNClassifier
except Exception as e:
    TabPFNClassifier = None
    print('TabPFN unavailable:', repr(e))
try:
    from causilo import CausiloClassifier
except Exception as e:
    CausiloClassifier = None
    print('Causilo unavailable:', repr(e))
try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None

warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(), '| polars', pl.__version__)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

All paths, hyperparameters, and environment flags are centralised here. The data directory is located automatically under `/kaggle/input`, so the notebook works regardless of the competition slug.

In [ ]:
class Settings:
    SEED       = 42
    GBDT_SEEDS = [42, 7, 2026]
    N_FOLDS    = 5
    _ON_KAGGLE = Path('/kaggle/input').exists()

    DATA_DIR   = (
        next(Path('/kaggle/input').rglob('train_applications.csv')).parent
        if _ON_KAGGLE else Path('../data')
    )
    OUTPUT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../output')
    FIG_DIR    = OUTPUT_DIR / 'figures'

    USE_LGB     = True
    USE_XGB     = True
    USE_CAT     = True
    USE_TABPFN  = True
    USE_CAUSILO = True

    N_GPU          = torch.cuda.device_count()
    FM_N_FEATURES  = 60
    FM_MAX_CONTEXT = 30000
    FM_N_EST       = 4
    FM_CHUNK       = 5000
    BLEND_STEP     = 0.1

    LGB_PARAMS = dict(
        objective='binary', metric='auc', learning_rate=0.02, num_leaves=31,
        min_child_samples=100, feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1,
        lambda_l2=5.0, verbose=-1, n_jobs=4,
    )
    XGB_PARAMS = dict(
        objective='binary:logistic', eval_metric='auc', learning_rate=0.02, max_depth=5,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0,
        tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    CAT_PARAMS = dict(
        iterations=5000, learning_rate=0.03, depth=6, l2_leaf_reg=5, loss_function='Logloss',
        task_type='GPU' if torch.cuda.is_available() else 'CPU', od_type='Iter', od_wait=200,
        use_best_model=True, verbose=0,
    )
    N_ROUNDS   = 5000
    EARLY_STOP = 200

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)
print('data:', CFG.DATA_DIR)

The TabPFN-3.5 weights are gated, so the access token is read from Kaggle secrets when it exists. Missing secrets are not an error.

In [ ]:
if UserSecretsClient is not None:
    for name in ['TABPFN_TOKEN', 'HF_TOKEN']:
        try:
            os.environ[name] = UserSecretsClient().get_secret(name)
            print('loaded secret', name)
        except Exception:
            pass
if 'TABPFN_TOKEN' not in os.environ and 'HF_TOKEN' in os.environ:
    os.environ['TABPFN_TOKEN'] = os.environ['HF_TOKEN']

## Load Dataset

All seven tables are read with Polars and dates are parsed on load. No visualisation here.

In [ ]:
TABLES = ['train_applications', 'test_applications', 'customer_profiles', 'previous_applications',
          'repayment_history', 'account_behavior', 'bureau_observations', 'ecommerce_transactions']
T = {t: pl.read_csv(CFG.DATA_DIR / f'{t}.csv', try_parse_dates=True, infer_schema_length=100000) for t in TABLES}
sample_sub = pl.read_csv(CFG.DATA_DIR / 'sample_submission.csv')
for t, df in T.items():
    print(f'{t:<24} {df.shape}')

---

# Exploratory Data Analysis <a name="3"></a>

---

The analysis below reproduces the key findings of the offline EDA that shaped the feature set. All statistics use the competition KS definition, so the helpers are defined first and reused throughout the notebook.

## KS Utilities

`ks_score` matches the official scorer (`ks_2samp` on bad versus good scores). `ks_nan` treats missing values as their own bin at either extreme, which mirrors how tree models route missing values. `psi` measures train-to-test drift on train deciles.

In [ ]:
def ks_score(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    return float(ks_2samp(s[y == 1], s[y == 0]).statistic)

def ks_nan(y, s):
    s = np.asarray(s, dtype=float)
    if np.isnan(s).all():
        return 0.0
    lo, hi = np.nanmin(s) - 1, np.nanmax(s) + 1
    return max(ks_score(y, np.where(np.isnan(s), lo, s)), ks_score(y, np.where(np.isnan(s), hi, s)))

def psi(a, b, bins=10):
    a, b = np.asarray(a, float), np.asarray(b, float)
    qs = np.unique(np.nanquantile(a, np.linspace(0, 1, bins + 1))) if (~np.isnan(a)).any() else np.array([0.0])
    def dist(x):
        idx = np.where(np.isnan(x), -1, np.searchsorted(qs[1:-1], x, side='right'))
        cnt = np.array([(idx == k).sum() for k in range(-1, len(qs))], float) + 0.5
        return cnt / cnt.sum()
    pa, pb = dist(a), dist(b)
    return float(((pb - pa) * np.log(pb / pa)).sum())

## Target and Time

The target rate and the monthly volume of both splits show whether a time-based split is required.

In [ ]:
tr_raw, te_raw = T['train_applications'], T['test_applications']
print(f"train bad rate: {tr_raw['target'].mean():.4f}  (n_bad={tr_raw['target'].sum()})")
m = pl.concat([
    tr_raw.select(ym=pl.col('decision_datetime').dt.strftime('%Y-%m'), target='target', split=pl.lit('train')),
    te_raw.select(ym=pl.col('decision_datetime').dt.strftime('%Y-%m'), target=pl.lit(None, pl.Int64), split=pl.lit('test')),
])
tm = m.group_by('ym', 'split').agg(n=pl.len(), rate=pl.col('target').mean()).sort('ym')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
p = tm.filter(pl.col('split') == 'train')
ax[0].bar(p['ym'], p['rate']); ax[0].set_title('train bad rate by decision month'); ax[0].tick_params(axis='x', rotation=60)
for s in ['train', 'test']:
    q = tm.filter(pl.col('split') == s)
    ax[1].plot(q['ym'], q['n'] / q['n'].sum(), marker='o', label=s)
ax[1].legend(); ax[1].set_title('share of applications by month'); ax[1].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'target_time.png', dpi=90); plt.show()

#### Insights

> The bad rate is 13.6% and stays between 12% and 15% in every month of 2025. Train and test applications are interleaved over the same twelve months in the same 70/30 proportion, so the test set is a random sample of the same period rather than an out-of-time holdout.

> Stratified K-fold therefore mimics the leaderboard split, and no time-based validation is needed.

## Risk-Based Pricing

The interest rate takes only five values. If the lender priced loans from a risk grade, the rate itself carries that grade.

In [ ]:
pricing = (tr_raw.group_by('annual_interest_rate')
           .agg(n=pl.len(), bad_rate=pl.col('target').mean()).sort('annual_interest_rate'))
print(pricing)
print('univariate KS of annual_interest_rate:', round(ks_score(tr_raw['target'], tr_raw['annual_interest_rate']), 4))

#### Insights

> The bad rate rises monotonically from 5.5% at a 16% rate to 50% at a 32% rate, a univariate KS of 0.23. The offline analysis shows that the mean bureau score falls by about 40 points per tier, so the rate is largely a coarse copy of the bureau score. Inside bureau-score deciles the rate keeps only about 0.06 KS.

## Pre-Application Spending Burst

E-commerce orders are counted per customer per week before the decision, separately for bad and good loans. The shape of this curve sets the aggregation windows.

In [ ]:
o = (T['ecommerce_transactions'].join(tr_raw.select('customer_id', 'decision_datetime', 'target'), on='customer_id')
     .group_by('order_id').agg(pl.col('customer_id', 'decision_datetime', 'transaction_datetime', 'target').first())
     .with_columns(wk=((pl.col('decision_datetime') - pl.col('transaction_datetime')).dt.total_days() // 7)))
ncls = tr_raw.group_by('target').agg(ncust=pl.len())
wk = (o.filter(pl.col('wk') < 52).group_by('wk', 'target').agg(n=pl.len()).join(ncls, on='target')
      .with_columns(per=pl.col('n') / pl.col('ncust')).sort('wk'))
fig, ax = plt.subplots(figsize=(9, 4))
for k in (0, 1):
    q = wk.filter(pl.col('target') == k)
    ax.plot(q['wk'], q['per'], label=f'target={k}')
ax.invert_xaxis(); ax.set_xlabel('weeks before decision'); ax.set_ylabel('orders per customer'); ax.legend()
ax.set_title('E-commerce orders per customer per week')
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'ecom_burst.png', dpi=90); plt.show()
r = wk.pivot(on='target', index='wk', values='per').sort('wk').head(6)
print(r.with_columns(bad_over_good=pl.col('1') / pl.col('0')))

#### Insights

> Bad and good customers order at the same rate until about 16 weeks before the decision. After that, bad customers accelerate and reach 2.3 times the good rate in the final week. Lifetime totals dilute this signal, so the features use 7 to 90 day windows and an acceleration ratio against a 17-52 week baseline.

> Account behaviour shows the same pattern. The outflow-to-inflow ratio and low-balance days of bad customers deteriorate over the last 6 months, although they look slightly healthier than good customers a year earlier. Recent-versus-baseline deltas capture this.

## Table Coverage

Coverage of each auxiliary table is compared between splits, because a coverage mismatch would make the CV score unrepresentative.

In [ ]:
split = pl.concat([tr_raw.select('customer_id', split=pl.lit('train')), te_raw.select('customer_id', split=pl.lit('test'))])
cov = []
for t in ['previous_applications', 'repayment_history', 'account_behavior', 'bureau_observations', 'ecommerce_transactions']:
    c = split.with_columns(has=pl.col('customer_id').is_in(T[t]['customer_id'].unique().implode()))
    cov.append(c.group_by('split').agg(pl.col('has').mean()).with_columns(table=pl.lit(t)))
print(pl.concat(cov).pivot(on='split', index='table', values='has'))

#### Insights

> Bureau and e-commerce data exist for every customer, while previous applications and account data exist for exactly 70% of customers in both splits (66% for repayment). The remaining 30% are thin-file customers, whose bad rate is lower (11.7% versus 14.4%).

> The offline audit also confirms that every auxiliary row is already dated before its customer's `decision_datetime`. There are no exact or near-duplicate customers across splits, and the hashed IDs carry no signal (KS < 0.01). The feature code still filters on the cutoff explicitly and is tested against injected future rows below.

---

# Feature Engineering <a name="4"></a>

---

Each auxiliary table is aggregated to one row per customer, and every builder filters on the customer's own cutoff (`decision_datetime`). Bureau reports must be *available* before the cutoff, account months must be *complete* before it, and e-commerce `order_total` is de-duplicated per order before it is summed, because it repeats on every item row.

## Feature Builders

The builders cover application and profile affordability ratios, bureau scores per provider and their change since earlier pulls, repayment lateness depth and active obligations, cash-flow levels and recent deltas, e-commerce recency and acceleration, and previous application outcomes and pricing.

In [ ]:
DAY = 86400.0


def _days(a, b):
    return (pl.col(a) - pl.col(b)).dt.total_seconds() / DAY


def app_features(apps: pl.DataFrame, prof: pl.DataFrame) -> pl.DataFrame:
    d = apps.join(prof, on="customer_id", how="left")
    return d.select(
        "customer_id",
        "loan_amount", "annual_interest_rate", "proposed_monthly_installment", "loan_purpose",
        "age", "region", "employment_type", "employment_tenure_months", "declared_monthly_income",
        "dependents", "housing_status",
        decision_lag_h=(pl.col("decision_datetime") - pl.col("application_datetime")).dt.total_seconds() / 3600,
        app_hour=pl.col("application_datetime").dt.hour(),
        app_dow=pl.col("application_datetime").dt.weekday(),
        dti_declared=pl.col("proposed_monthly_installment") / pl.col("declared_monthly_income"),
        lti_declared=pl.col("loan_amount") / pl.col("declared_monthly_income"),
        tenure_over_age=pl.col("employment_tenure_months") / (pl.col("age") * 12),
        income_per_head=pl.col("declared_monthly_income") / (1 + pl.col("dependents")),
    )


def bureau_features(apps: pl.DataFrame, bureau: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", sel_app=pl.col("application_id"), cutoff=pl.col("decision_datetime"))
    b = bureau.join(cut, on="customer_id").filter(pl.col("available_datetime") <= pl.col("cutoff"))
    b = b.with_columns(
        stale_d=_days("cutoff", "report_datetime"),
        delay_d=_days("available_datetime", "report_datetime"),
        is_sel=pl.col("application_id") == pl.col("sel_app"),
        prov=pl.col("bureau_provider").str.replace("Bureau_", "").str.to_lowercase(),
    )
    cur = b.filter("is_sel")
    num = ["score", "active_accounts", "outstanding_balance", "revolving_credit_limit",
           "revolving_outstanding_balance", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m"]
    agg = cur.group_by("customer_id").agg(
        bc_n_reports=pl.len(),
        bc_n_scored=pl.col("score").is_not_null().sum(),
        bc_score_mean=pl.col("score").mean(), bc_score_min=pl.col("score").min(),
        bc_score_max=pl.col("score").max(), bc_score_std=pl.col("score").std(),
        bc_active_max=pl.col("active_accounts").max(),
        bc_outstanding_max=pl.col("outstanding_balance").max(),
        bc_revlimit_max=pl.col("revolving_credit_limit").max(),
        bc_revbal_max=pl.col("revolving_outstanding_balance").max(),
        bc_util_max=pl.col("utilization").max(), bc_util_mean=pl.col("utilization").mean(),
        bc_inq_max=pl.col("inquiries_last_6m").max(), bc_inq_mean=pl.col("inquiries_last_6m").mean(),
        bc_delinq_max=pl.col("delinquent_accounts_last_12m").max(),
        bc_delinq_sum=pl.col("delinquent_accounts_last_12m").sum(),
        bc_stale_mean=pl.col("stale_d").mean(), bc_stale_max=pl.col("stale_d").max(),
        bc_delay_mean=pl.col("delay_d").mean(),
    )
    piv = cur.pivot(on="prov", index="customer_id", values=["score", "utilization", "inquiries_last_6m",
                                                            "delinquent_accounts_last_12m", "outstanding_balance"],
                    aggregate_function="first")
    piv = piv.rename({c: "bp_" + c.replace("inquiries_last_6m", "inq").replace("delinquent_accounts_last_12m", "delinq")
                      .replace("outstanding_balance", "out").replace("utilization", "util")
                      for c in piv.columns if c != "customer_id"})
    # history: bureau pulled at earlier (previous) applications
    hist = b.filter(~pl.col("is_sel")).group_by("customer_id").agg(
        bh_n_pulls=pl.col("application_id").n_unique(),
        bh_score_mean=pl.col("score").mean(), bh_score_min=pl.col("score").min(),
        bh_delinq_max=pl.col("delinquent_accounts_last_12m").max(),
        bh_inq_mean=pl.col("inquiries_last_6m").mean(),
        bh_last_score=pl.col("score").sort_by("report_datetime").drop_nulls().last(),
        bh_last_outstanding=pl.col("outstanding_balance").sort_by("report_datetime").last(),
        bh_pulls_180d=pl.col("application_id").filter(pl.col("stale_d") <= 180).n_unique(),
    )
    out = agg.join(piv, on="customer_id", how="left").join(hist, on="customer_id", how="left")
    return out.with_columns(
        bh_score_delta=pl.col("bc_score_mean") - pl.col("bh_last_score"),
        bh_outstanding_delta=pl.col("bc_outstanding_max") - pl.col("bh_last_outstanding"),
    )


def repayment_features(apps: pl.DataFrame, rep: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    r = rep.join(cut, on="customer_id").filter(pl.col("due_date") <= pl.col("cutoff"))
    r = r.with_columns(
        age_d=_days("cutoff", "due_date"),
        unpaid=pl.col("amount_due") - pl.col("amount_paid"),
        late_d=pl.when(pl.col("amount_paid") >= pl.col("amount_due")).then(_days("last_payment_datetime", "due_date")),
    )
    w = lambda days: pl.col("age_d") <= days  # noqa: E731
    agg = r.group_by("customer_id").agg(
        rp_n_loans=pl.col("loan_id").n_unique(), rp_n_inst=pl.len(),
        rp_maxdpd_max=pl.col("max_days_past_due").max(), rp_maxdpd_mean=pl.col("max_days_past_due").mean(),
        rp_n_dpd1=(pl.col("max_days_past_due") > 0).sum(), rp_n_dpd30=(pl.col("max_days_past_due") >= 30).sum(),
        rp_n_dpd60=(pl.col("max_days_past_due") >= 60).sum(), rp_n_dpd90=(pl.col("max_days_past_due") >= 90).sum(),
        rp_frac_dpd1=(pl.col("max_days_past_due") > 0).mean(), rp_frac_dpd30=(pl.col("max_days_past_due") >= 30).mean(),
        rp_cur_dpd_max=pl.col("days_past_due").max(), rp_n_unpaid=(pl.col("unpaid") > 0).sum(),
        rp_unpaid_amt=pl.col("unpaid").sum(),
        rp_paid_ratio=pl.col("amount_paid").sum() / pl.col("amount_due").sum(),
        rp_late_mean=pl.col("late_d").mean(), rp_late_max=pl.col("late_d").max(),
        rp_maxdpd_6m=pl.col("max_days_past_due").filter(w(183)).max(),
        rp_maxdpd_12m=pl.col("max_days_past_due").filter(w(365)).max(),
        rp_frac_dpd1_6m=(pl.col("max_days_past_due") > 0).filter(w(183)).mean(),
        rp_late_mean_6m=pl.col("late_d").filter(w(183)).mean(),
        rp_maxdpd_last3=pl.col("max_days_past_due").sort_by("due_date").tail(3).max(),
        rp_days_since_delinq=pl.col("age_d").filter(pl.col("max_days_past_due") > 0).min(),
        rp_days_since_due=pl.col("age_d").min(),
    )
    # active loans: fewer installments observed than contracted term -> remaining obligation
    loans = r.group_by("loan_id").agg(customer_id=pl.col("customer_id").first(), n_seen=pl.len(),
                                      inst=pl.col("amount_due").first())
    loans = loans.join(prev.select("loan_id", "term_months"), on="loan_id", how="left").with_columns(
        remaining=(pl.col("term_months") - pl.col("n_seen")).clip(lower_bound=0))
    act = loans.group_by("customer_id").agg(
        rp_n_active=(pl.col("remaining") > 0).sum(),
        rp_active_inst=pl.col("inst").filter(pl.col("remaining") > 0).sum(),
        rp_remaining_oblig=(pl.col("remaining") * pl.col("inst")).sum(),
    )
    return agg.join(act, on="customer_id", how="left")


def account_features(apps: pl.DataFrame, acc: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    a = acc.join(cut, on="customer_id").filter(
        pl.col("month").dt.offset_by("1mo").cast(pl.Datetime) <= pl.col("cutoff")).sort("customer_id", "month")
    a = a.with_columns(mago=pl.col("month").rank("dense", descending=True).over("customer_id"))
    l3 = pl.col("mago") <= 3
    out = a.group_by("customer_id").agg(
        ac_n_months=pl.len(),
        ac_inflow_mean=pl.col("inflows").mean(), ac_outflow_mean=pl.col("outflows").mean(),
        ac_income_mean=pl.col("income_like_credits").mean(), ac_income_std=pl.col("income_like_credits").std(),
        ac_income_min=pl.col("income_like_credits").min(),
        ac_avgbal_mean=pl.col("average_balance").mean(), ac_minbal_min=pl.col("minimum_balance").min(),
        ac_minbal_mean=pl.col("minimum_balance").mean(),
        ac_lowdays_mean=pl.col("low_balance_days").mean(), ac_lowdays_max=pl.col("low_balance_days").max(),
        ac_frac_zero_min=(pl.col("minimum_balance") == 0).mean(),
        ac_net_flow_mean=(pl.col("inflows") - pl.col("outflows")).mean(),
        ac_out_in_ratio=pl.col("outflows").sum() / pl.col("inflows").sum(),
        ac_income_l3=pl.col("income_like_credits").filter(l3).mean(),
        ac_avgbal_l3=pl.col("average_balance").filter(l3).mean(),
        ac_lowdays_l3=pl.col("low_balance_days").filter(l3).mean(),
        ac_out_in_l3=pl.col("outflows").filter(l3).sum() / pl.col("inflows").filter(l3).sum(),
        ac_last_close=pl.col("closing_balance").last(),
        ac_out_in_b=pl.col("outflows").filter(pl.col("mago").is_between(7, 12)).sum()
        / pl.col("inflows").filter(pl.col("mago").is_between(7, 12)).sum(),
        ac_lowdays_b=pl.col("low_balance_days").filter(pl.col("mago").is_between(7, 12)).mean(),
        ac_avgbal_b=pl.col("average_balance").filter(pl.col("mago").is_between(7, 12)).mean(),
        ac_out_in_l1=(pl.col("outflows") / pl.col("inflows")).filter(pl.col("mago") == 1).first(),
        ac_close_slope=pl.cov(pl.col("mago").cast(pl.Float64), pl.col("closing_balance").cast(pl.Float64))
        / pl.col("mago").cast(pl.Float64).var() * -1,
    )
    return out.with_columns(
        ac_income_cv=pl.col("ac_income_std") / pl.col("ac_income_mean"),
        ac_income_trend=pl.col("ac_income_l3") / pl.col("ac_income_mean"),
        ac_bal_trend=pl.col("ac_avgbal_l3") / (pl.col("ac_avgbal_mean") + 1),
        ac_out_in_delta=pl.col("ac_out_in_l3") - pl.col("ac_out_in_b"),
        ac_lowdays_delta=pl.col("ac_lowdays_l3") - pl.col("ac_lowdays_b"),
        ac_bal_delta=(pl.col("ac_avgbal_l3") + 1) / (pl.col("ac_avgbal_b") + 1),
    )


def ecommerce_features(apps: pl.DataFrame, ec: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    e = ec.join(cut, on="customer_id").filter(pl.col("transaction_datetime") <= pl.col("cutoff"))
    e = e.with_columns(age_d=_days("cutoff", "transaction_datetime"), done=pl.col("order_status") == "completed")
    # order level: order_total repeats on each item -> dedupe before aggregating
    o = e.group_by("order_id").agg(pl.col("customer_id", "age_d", "order_total", "payment_method", "done").first())
    agg = o.group_by("customer_id").agg(
        ec_n_orders=pl.len(), ec_cancel_rate=1 - pl.col("done").mean(),
        ec_spend=pl.col("order_total").filter("done").sum(),
        ec_aov=pl.col("order_total").filter("done").mean(),
        ec_recency_d=pl.col("age_d").min(), ec_tenure_d=pl.col("age_d").max(),
        ec_orders_7d=(pl.col("age_d") <= 7).sum(), ec_orders_14d=(pl.col("age_d") <= 14).sum(),
        ec_orders_30d=(pl.col("age_d") <= 30).sum(), ec_orders_60d=(pl.col("age_d") <= 60).sum(),
        ec_orders_90d=(pl.col("age_d") <= 90).sum(),
        ec_base_wk=((pl.col("age_d") > 120) & (pl.col("age_d") <= 365)).sum()
        / ((pl.col("age_d").max().clip(upper_bound=365) - 120).clip(lower_bound=7) / 7),
        ec_spend_30d=pl.col("order_total").filter(pl.col("done") & (pl.col("age_d") <= 30)).sum(),
        ec_spend_90d=pl.col("order_total").filter(pl.col("done") & (pl.col("age_d") <= 90)).sum(),
        ec_cancel_90d=(~pl.col("done")).filter(pl.col("age_d") <= 90).sum(),
        **{f"ec_pm_{m}": (pl.col("payment_method") == m).mean()
           for m in ["COD", "e_wallet", "bank_transfer", "debit_card", "credit_card"]},
    )
    cat = (e.filter("done").group_by("customer_id", "product_category").agg(s=pl.col("item_total").sum())
           .with_columns(s=pl.col("s") / pl.col("s").sum().over("customer_id"))
           .pivot(on="product_category", index="customer_id", values="s"))
    cat = cat.rename({c: f"ec_cat_{c}" for c in cat.columns if c != "customer_id"})
    out = agg.join(cat, on="customer_id", how="left").with_columns(
        ec_spend_per_month=pl.col("ec_spend") / (pl.col("ec_tenure_d") / 30.4 + 1),
        ec_spend_90d_ratio=pl.col("ec_spend_90d") / 3 / (pl.col("ec_spend") / (pl.col("ec_tenure_d") / 30.4 + 1) + 1),
    )
    return out.with_columns(
        pl.col("^ec_cat_.*$").fill_null(0),
        ec_accel_30d=(pl.col("ec_orders_30d") / (30 / 7)) / (pl.col("ec_base_wk") + 0.1),
        ec_accel_90d=(pl.col("ec_orders_90d") / (90 / 7)) / (pl.col("ec_base_wk") + 0.1),
    )


def prev_features(apps: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"), cur_rate=pl.col("annual_interest_rate"),
                      cur_amt=pl.col("loan_amount"))
    p = prev.join(cut, on="customer_id").filter(pl.col("decision_datetime") <= pl.col("cutoff"))
    p = p.with_columns(age_d=_days("cutoff", "application_datetime"))
    out = p.group_by("customer_id").agg(
        pv_n=pl.len(), pv_n_appr=(pl.col("decision") == "approved").sum(),
        pv_n_rej=(pl.col("decision") == "rejected").sum(), pv_n_canc=(pl.col("decision") == "cancelled").sum(),
        pv_rej_rate=(pl.col("decision") == "rejected").mean(),
        pv_days_since_last=pl.col("age_d").min(), pv_days_since_first=pl.col("age_d").max(),
        pv_n_90d=(pl.col("age_d") <= 90).sum(), pv_n_180d=(pl.col("age_d") <= 180).sum(),
        pv_rej_365d=((pl.col("decision") == "rejected") & (pl.col("age_d") <= 365)).sum(),
        pv_rate_mean=pl.col("annual_interest_rate").mean(), pv_rate_max=pl.col("annual_interest_rate").max(),
        pv_rate_last=pl.col("annual_interest_rate").sort_by("application_datetime").drop_nulls().last(),
        pv_appr_ratio=(pl.col("approved_amount") / pl.col("requested_amount")).mean(),
        pv_req_mean=pl.col("requested_amount").mean(), pv_appr_sum=pl.col("approved_amount").sum(),
        pv_appr_max=pl.col("approved_amount").max(),
        pv_last_decision_rej=(pl.col("decision").sort_by("application_datetime").last() == "rejected"),
        cur_rate=pl.col("cur_rate").first(), cur_amt=pl.col("cur_amt").first(),
    )
    return out.with_columns(
        pv_rate_delta=pl.col("cur_rate") - pl.col("pv_rate_last"),
        pv_amt_vs_max=pl.col("cur_amt") / pl.col("pv_appr_max"),
        pv_last_decision_rej=pl.col("pv_last_decision_rej").cast(pl.Int8),
    ).drop("cur_rate", "cur_amt")


def build_all(tables: dict) -> pl.DataFrame:
    tr = tables["train_applications"].with_columns(is_train=pl.lit(1))
    te = tables["test_applications"].with_columns(target=pl.lit(None, pl.Int64), is_train=pl.lit(0))
    apps = pl.concat([tr, te], how="diagonal_relaxed")
    f = apps.select("application_id", "customer_id", "decision_datetime", "target", "is_train")
    for part in [
        app_features(apps, tables["customer_profiles"]),
        bureau_features(apps, tables["bureau_observations"]),
        repayment_features(apps, tables["repayment_history"], tables["previous_applications"]),
        account_features(apps, tables["account_behavior"]),
        ecommerce_features(apps, tables["ecommerce_transactions"]),
        prev_features(apps, tables["previous_applications"]),
    ]:
        f = f.join(part, on="customer_id", how="left")
    # cross-table ratios: the real affordability picture
    f = f.with_columns(
        has_history=pl.col("pv_n").is_not_null().cast(pl.Int8),
        dti_verified=pl.col("proposed_monthly_installment") / pl.col("ac_income_mean"),
        dti_total=(pl.col("proposed_monthly_installment") + pl.col("rp_active_inst").fill_null(0))
        / pl.col("declared_monthly_income"),
        income_verif_ratio=pl.col("ac_income_mean") / pl.col("declared_monthly_income"),
        debt_to_income=pl.col("bc_outstanding_max") / pl.col("declared_monthly_income"),
        inst_to_avgbal=pl.col("proposed_monthly_installment") / (pl.col("ac_avgbal_mean") + 1),
        ecspend_to_income=pl.col("ec_spend_per_month") / pl.col("declared_monthly_income"),
    )
    return f

In [ ]:
t0 = time.time()
feats = build_all(T)
assert feats['application_id'].n_unique() == feats.height == T['train_applications'].height + T['test_applications'].height
print(f'feature matrix {feats.shape} built in {time.time() - t0:.1f}s')

## Leakage Injection Test

Fake rows dated after each customer's cutoff are appended to four tables, using extreme values (DPD of 999, a score of 300, a 1e9 order, and the incomplete decision month). If any feature changes, the cutoff logic is broken and the notebook stops.

In [ ]:
apps_all = pl.concat([T['train_applications'], T['test_applications']], how='diagonal_relaxed').select('customer_id', 'decision_datetime')
fut = apps_all.sample(2000, seed=1)
def first_rows(t):
    return T[t].join(fut, on='customer_id').group_by('customer_id').first()
inj = {
    'repayment_history': first_rows('repayment_history').with_columns(
        due_date=pl.col('decision_datetime') + pl.duration(days=5), max_days_past_due=pl.lit(999), days_past_due=pl.lit(999)),
    'ecommerce_transactions': first_rows('ecommerce_transactions').with_columns(
        transaction_datetime=pl.col('decision_datetime') + pl.duration(hours=1),
        order_id=pl.lit('O_FAKE') + pl.col('customer_id'), order_total=pl.lit(10**9)),
    'bureau_observations': first_rows('bureau_observations').with_columns(
        available_datetime=pl.col('decision_datetime') + pl.duration(days=5), score=pl.lit(300.0)),
    'account_behavior': first_rows('account_behavior').with_columns(
        month=pl.col('decision_datetime').dt.truncate('1mo').cast(pl.Date), outflows=pl.lit(10**10)),
}
T_leak = dict(T)
for t, extra in inj.items():
    T_leak[t] = pl.concat([T[t], extra.select(T[t].columns)], how='vertical_relaxed')
f_leak = build_all(T_leak).sort('application_id')
f_ref = feats.sort('application_id')
num_cols = [c for c, d in f_ref.schema.items() if d.is_numeric()]
changed = [c for c in num_cols
           if not f_ref[c].cast(pl.Float64).fill_nan(-7).fill_null(-7).equals(f_leak[c].cast(pl.Float64).fill_nan(-7).fill_null(-7))]
assert not changed, f'cutoff leak in {changed}'
print('leakage injection test passed: no feature moved')
del T_leak, f_leak, f_ref; gc.collect()

## Feature Diagnostics

Every numeric feature is scored by univariate KS against the target and by PSI between train and test. A high-KS feature that also drifts would split public and private scores, so both are checked together.

In [ ]:
ID_COLS  = ['application_id', 'customer_id', 'decision_datetime', 'target', 'is_train']
CAT_COLS = ['loan_purpose', 'region', 'employment_type', 'housing_status']
FEATURES = sorted(c for c in feats.columns if c not in ID_COLS)
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

tr_f, te_f = feats.filter(pl.col('is_train') == 1), feats.filter(pl.col('is_train') == 0)
y = tr_f['target'].to_numpy()
diag = pl.DataFrame({
    'feature': NUM_COLS,
    'ks': [ks_nan(y, tr_f[c].cast(pl.Float64).to_numpy()) for c in NUM_COLS],
    'psi': [psi(tr_f[c].cast(pl.Float64).to_numpy(), te_f[c].cast(pl.Float64).to_numpy()) for c in NUM_COLS],
    'coverage': [1 - tr_f[c].null_count() / tr_f.height for c in NUM_COLS],
}).sort('ks', descending=True)
with pl.Config(tbl_rows=25, float_precision=4):
    print(diag.head(25))
print('max PSI over all features:', round(diag['psi'].max(), 5))
print(f'{len(FEATURES)} features ({len(CAT_COLS)} categorical)')

#### Insights

> The strongest single features are repayment lateness (`rp_frac_dpd1`, KS about 0.32), e-commerce acceleration (`ec_accel_30d`, about 0.31) and the mean bureau score (about 0.25). The first two are nearly orthogonal to the bureau score and keep about 0.30 KS inside score deciles, which is why combining sources gives a large lift. A rank average of nine such features with no training already reaches a KS of about 0.41.

> The maximum PSI is far below the usual 0.1 alert level, so no feature needs to be dropped for stability.

---

# Validation Strategy <a name="5"></a>

---

Because the test set is a random sample of the training period, repeated stratified 5-fold CV is used. Every model reports per-fold KS, the out-of-fold (OOF) KS and its spread. The OOF predictions also drive a public/private split simulation that estimates how far the two leaderboards can disagree from sampling noise alone.

In [ ]:
X_tr = tr_f.select(FEATURES).to_pandas()
X_te = te_f.select(FEATURES).to_pandas()
for c in CAT_COLS:
    cats = pd.Categorical(pd.concat([X_tr[c], X_te[c]])).categories
    X_tr[c] = pd.Categorical(X_tr[c], categories=cats)
    X_te[c] = pd.Categorical(X_te[c], categories=cats)
test_ids = te_f['application_id'].to_numpy()

def folds(seed):
    return list(StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=seed).split(X_tr, y))

def gap_simulation(pred, n_test=30000, public_share=0.5, n_iter=300, seed=0):
    rng = np.random.default_rng(seed)
    gaps = []
    for _ in range(n_iter):
        idx = rng.choice(len(y), n_test, replace=False)
        cut = int(public_share * n_test)
        gaps.append(ks_score(y[idx[:cut]], pred[idx[:cut]]) - ks_score(y[idx[cut:]], pred[idx[cut:]]))
    g = np.abs(np.array(gaps))
    return float(np.median(g)), float(np.percentile(g, 95))

RESULTS = {}

def report(name, oof, fold_ks, test_pred):
    RESULTS[name] = dict(oof=oof, test=test_pred, fold_ks=fold_ks)
    med, p95 = gap_simulation(oof)
    print(f'[{name}] OOF KS={ks_score(y, oof):.4f}  AUC={roc_auc_score(y, oof):.4f}  '
          f'fold KS={np.mean(fold_ks):.4f}+-{np.std(fold_ks):.4f}  sim |pub-priv| p50={med:.4f} p95={p95:.4f}')
    np.save(CFG.OUTPUT_DIR / f'oof_{name}.npy', oof)
    np.save(CFG.OUTPUT_DIR / f'test_{name}.npy', test_pred)

---

# Gradient Boosting Models <a name="6"></a>

---

The three gradient boosting libraries are run over the same repeated folds (3 seeds x 5 folds). Each one handles missing values natively, which matters because 30% of customers lack the history tables entirely. Predictions are averaged over seeds for OOF rows and over all 15 models for the test set.

In [ ]:
def run_gbdt(name, fit_predict):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_te))
    fold_ks = []
    t0 = time.time()
    for seed in CFG.GBDT_SEEDS:
        for k, (tri, vai) in enumerate(folds(seed)):
            pv, pt = fit_predict(tri, vai, seed)
            oof[vai] += pv / len(CFG.GBDT_SEEDS)
            test_pred += pt / (len(CFG.GBDT_SEEDS) * CFG.N_FOLDS)
            fold_ks.append(ks_score(y[vai], pv))
        print(f'  {name} seed {seed}: fold KS {np.round(fold_ks[-CFG.N_FOLDS:], 4)}  ({time.time() - t0:.0f}s)')
    report(name, oof, fold_ks, test_pred)

## LightGBM

LightGBM uses native categorical splits and early stopping on validation AUC, which is monotonically related to KS for a fixed ranking quality.

In [ ]:
LGB_MODELS = []

def lgb_fit(tri, vai, seed):
    dtr = lgb.Dataset(X_tr.iloc[tri], y[tri], categorical_feature=CAT_COLS)
    dva = lgb.Dataset(X_tr.iloc[vai], y[vai], categorical_feature=CAT_COLS, reference=dtr)
    m = lgb.train({**CFG.LGB_PARAMS, 'seed': seed}, dtr, CFG.N_ROUNDS, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(CFG.EARLY_STOP, verbose=False)])
    LGB_MODELS.append(m)
    return m.predict(X_tr.iloc[vai], num_iteration=m.best_iteration), m.predict(X_te, num_iteration=m.best_iteration)

if CFG.USE_LGB:
    run_gbdt('lgb', lgb_fit)

The gain importance of the LightGBM models shows which feature families carry the ranking.

In [ ]:
if LGB_MODELS:
    imp = pd.DataFrame({'feature': LGB_MODELS[0].feature_name(),
                        'gain': np.mean([m.feature_importance('gain') for m in LGB_MODELS], axis=0)})
    imp = imp.sort_values('gain', ascending=False).head(30)
    fig, ax = plt.subplots(figsize=(8, 9))
    ax.barh(imp['feature'][::-1], imp['gain'][::-1])
    ax.set_title('LightGBM mean gain importance (top 30)')
    plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'lgb_importance.png', dpi=90); plt.show()

## XGBoost

XGBoost runs the histogram algorithm on the GPU with native categorical support.

In [ ]:
D_TE = xgb.DMatrix(X_te, enable_categorical=True)

def xgb_fit(tri, vai, seed):
    dtr = xgb.DMatrix(X_tr.iloc[tri], y[tri], enable_categorical=True)
    dva = xgb.DMatrix(X_tr.iloc[vai], y[vai], enable_categorical=True)
    m = xgb.train({**CFG.XGB_PARAMS, 'seed': seed}, dtr, CFG.N_ROUNDS, evals=[(dva, 'va')],
                  early_stopping_rounds=CFG.EARLY_STOP, verbose_eval=False)
    it = (0, m.best_iteration + 1)
    return m.predict(dva, iteration_range=it), m.predict(D_TE, iteration_range=it)

if CFG.USE_XGB:
    run_gbdt('xgb', xgb_fit)

## CatBoost

CatBoost trains on the GPU with ordered target statistics for the four categorical columns, which are passed as strings.

In [ ]:
XC_TR, XC_TE = X_tr.copy(), X_te.copy()
for c in CAT_COLS:
    XC_TR[c] = XC_TR[c].astype(str)
    XC_TE[c] = XC_TE[c].astype(str)

def cat_fit(tri, vai, seed):
    m = CatBoostClassifier(**CFG.CAT_PARAMS, random_seed=seed)
    m.fit(Pool(XC_TR.iloc[tri], y[tri], cat_features=CAT_COLS),
          eval_set=Pool(XC_TR.iloc[vai], y[vai], cat_features=CAT_COLS))
    return m.predict_proba(XC_TR.iloc[vai])[:, 1], m.predict_proba(XC_TE)[:, 1]

if CFG.USE_CAT:
    run_gbdt('cat', cat_fit)

#### Insights

> Compare each model's OOF KS against the model-free baseline of about 0.41. The fold-to-fold standard deviation and the simulated public/private gap show how much of any leaderboard movement is just sampling noise. A difference between models smaller than the fold standard deviation should not drive a decision.

---

# Tabular Foundation Models <a name="7"></a>

---

TabPFN-3.5 and Causilo are pretrained transformers that predict in-context: the training rows are the prompt, and no gradient step is taken on this data. They bring a different inductive bias from tree ensembles, which makes them valuable blend members. LimiX-2 was considered as well, but it requires flash-attention and torch 2.9+, which the T4 (Turing) GPUs cannot run, so it is left out.

## Feature Subset and Context

Transformers pay for every column, and many repayment features are near-copies of each other (Spearman correlation above 0.95). Inside each fold, numeric features are ranked by univariate KS on the fold's training part only, which avoids selection leakage. Near-duplicates are then dropped greedily, and the top `FM_N_FEATURES` are kept with the categorical codes. The context is a stratified subsample capped at `FM_MAX_CONTEXT` rows. On a CUDA out-of-memory error the cap halves automatically.

In [ ]:
def select_fm_features(idx):
    sub, yy = X_tr.iloc[idx], y[idx]
    ranked = sorted(NUM_COLS, key=lambda c: -ks_nan(yy, sub[c].to_numpy(dtype=float)))[:2 * CFG.FM_N_FEATURES]
    corr = sub[ranked].rank().corr().abs()
    keep = []
    for c in ranked:
        if all(corr.loc[c, k] < 0.95 for k in keep):
            keep.append(c)
        if len(keep) == CFG.FM_N_FEATURES:
            break
    return keep + CAT_COLS

def fm_matrix(df, cols):
    out = df[cols].copy()
    for c in CAT_COLS:
        out[c] = out[c].cat.codes
    return out.to_numpy(dtype=np.float32)

def stratified_subsample(idx, n, seed):
    if len(idx) <= n:
        return idx
    rng = np.random.default_rng(seed)
    pos, neg = idx[y[idx] == 1], idx[y[idx] == 0]
    k = int(round(n * len(pos) / len(idx)))
    return np.r_[rng.choice(pos, k, replace=False), rng.choice(neg, n - k, replace=False)]

def predict_chunks(m, X):
    return np.concatenate([m.predict_proba(X[i:i + CFG.FM_CHUNK])[:, 1] for i in range(0, len(X), CFG.FM_CHUNK)])

def run_fm(name, make_model):
    oof, test_pred, fold_ks = np.zeros(len(y)), np.zeros(len(X_te)), []
    ctx, t0 = CFG.FM_MAX_CONTEXT, time.time()
    for k, (tri, vai) in enumerate(folds(CFG.SEED)):
        cols = select_fm_features(tri)
        cat_idx = [cols.index(c) for c in CAT_COLS]
        Xv, Xt = fm_matrix(X_tr.iloc[vai], cols), fm_matrix(X_te, cols)
        while True:
            ci, m = stratified_subsample(tri, ctx, CFG.SEED + k), None
            try:
                m = make_model(cat_idx)
                m.fit(fm_matrix(X_tr.iloc[ci], cols), y[ci])
                pv, pt = predict_chunks(m, Xv), predict_chunks(m, Xt)
                break
            except torch.cuda.OutOfMemoryError:
                del m; gc.collect(); torch.cuda.empty_cache()
                ctx //= 2
                print(f'  OOM -> context reduced to {ctx}')
                if ctx < 2000:
                    raise
        oof[vai], test_pred = pv, test_pred + pt / CFG.N_FOLDS
        fold_ks.append(ks_score(y[vai], pv))
        print(f'  {name} fold {k}: KS={fold_ks[-1]:.4f} ctx={len(ci)} feats={len(cols)} ({time.time() - t0:.0f}s)')
        del m; gc.collect(); torch.cuda.empty_cache()
    report(name, oof, fold_ks, test_pred)

## TabPFN-3.5

TabPFN-3.5 runs on the first GPU with categorical columns flagged explicitly. A failure (for example, missing gated-weight access) is reported and skipped so that the notebook still completes.

In [ ]:
if CFG.USE_TABPFN and TabPFNClassifier is not None and torch.cuda.is_available():
    try:
        run_fm('tabpfn', lambda cat_idx: TabPFNClassifier(
            n_estimators=CFG.FM_N_EST, device='cuda:0', categorical_features_indices=cat_idx,
            random_state=CFG.SEED))
    except Exception as e:
        print('TabPFN skipped:', repr(e))

## Causilo

Causilo runs on the second GPU when two are available. It receives the same feature subset and context as TabPFN.

In [ ]:
if CFG.USE_CAUSILO and CausiloClassifier is not None and torch.cuda.is_available():
    try:
        dev = 'cuda:1' if CFG.N_GPU > 1 else 'cuda:0'
        run_fm('causilo', lambda cat_idx: CausiloClassifier(
            n_estimators=CFG.FM_N_EST, random_state=CFG.SEED, device=dev))
    except Exception as e:
        print('Causilo skipped:', repr(e))

---

# Ensemble and KS Stability <a name="8"></a>

---

KS depends only on ranks, so each model's predictions are converted to normalised ranks before blending. Blend weights are searched on a simplex grid over OOF KS. To guard against fitting weights to noise, the search is cross-checked: weights fitted on one half of the OOF rows are scored on the other half. The searched weights replace equal weights only if they win by more than 0.002 KS.

In [ ]:
names = list(RESULTS)
R_OOF = np.column_stack([rankdata(RESULTS[n]['oof']) / len(y) for n in names])
R_TE = np.column_stack([rankdata(RESULTS[n]['test']) / len(X_te) for n in names])
print(pd.DataFrame(R_OOF, columns=names).corr().round(3))

grid = [np.array(w) for w in product(np.arange(0, 1 + 1e-9, CFG.BLEND_STEP), repeat=len(names))
        if abs(sum(w) - 1) < 1e-6]

def best_weights(rows):
    return max(grid, key=lambda w: ks_score(y[rows], R_OOF[rows] @ w))

w_eq = np.ones(len(names)) / len(names)
w_best = best_weights(np.arange(len(y)))
half = np.random.default_rng(0).permutation(len(y))
a, b = half[: len(y) // 2], half[len(y) // 2:]
w_a, w_b = best_weights(a), best_weights(b)
print('cross-checked searched weights KS:', round(ks_score(y[b], R_OOF[b] @ w_a), 4), round(ks_score(y[a], R_OOF[a] @ w_b), 4),
      '| equal weights on same halves:', round(ks_score(y[b], R_OOF[b] @ w_eq), 4), round(ks_score(y[a], R_OOF[a] @ w_eq), 4))
gain = (ks_score(y[b], R_OOF[b] @ w_a) + ks_score(y[a], R_OOF[a] @ w_b)
        - ks_score(y[b], R_OOF[b] @ w_eq) - ks_score(y[a], R_OOF[a] @ w_eq)) / 2
W = w_best if gain > 0.002 else w_eq
print('weights:', dict(zip(names, np.round(W, 2))), f'(held-out gain of search {gain:+.4f})')

blend_oof, blend_te = R_OOF @ W, R_TE @ W
for n in names:
    print(f'  {n:<8} OOF KS {ks_score(y, RESULTS[n]["oof"]):.4f}')
print(f'  BLEND    OOF KS {ks_score(y, blend_oof):.4f}')

## KS Stability Checks

The blend is checked along four axes: per-fold KS, the simulated public/private gap at 30% and 50% public shares, KS inside the thin-file and history segments, and KS per decision month. A stable model shows small spreads on all four.

In [ ]:
fk = [ks_score(y[v], blend_oof[v]) for _, v in folds(CFG.SEED)]
print(f'blend fold KS: {np.round(fk, 4)}  mean={np.mean(fk):.4f} sd={np.std(fk):.4f}')
for share in (0.3, 0.5):
    med, p95 = gap_simulation(blend_oof, public_share=share)
    print(f'public share {share:.0%}: simulated |KS_pub - KS_priv| median={med:.4f} p95={p95:.4f}')
hist = tr_f['has_history'].to_numpy()
for g, lab in [(0, 'thin-file'), (1, 'with history')]:
    msk = hist == g
    print(f'{lab:<13} n={msk.sum():>6} bad={y[msk].mean():.3f} KS={ks_score(y[msk], blend_oof[msk]):.4f}')
month = tr_f['decision_datetime'].dt.month().to_numpy()
mk = [ks_score(y[month == mth], blend_oof[month == mth]) for mth in range(1, 13)]
print('KS by decision month:', np.round(mk, 3), f'sd={np.std(mk):.4f}')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
s = np.sort(blend_oof)
f1 = np.searchsorted(np.sort(blend_oof[y == 1]), s, side='right') / (y == 1).sum()
f0 = np.searchsorted(np.sort(blend_oof[y == 0]), s, side='right') / (y == 0).sum()
i = np.argmax(np.abs(f1 - f0))
ax[0].plot(s, f1, label='bad (F1)'); ax[0].plot(s, f0, label='good (F0)')
ax[0].vlines(s[i], f1[i], f0[i], color='k', ls='--', label=f'KS={abs(f1[i] - f0[i]):.4f}')
ax[0].set_title('Blend OOF score CDFs'); ax[0].legend()
ax[1].bar(range(1, 13), mk); ax[1].set_title('Blend OOF KS by decision month'); ax[1].set_xlabel('month')
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'ks_stability.png', dpi=90); plt.show()

#### Insights

> A simulated public/private gap of about 0.01 is the noise floor for a 30,000-row test set with about 4,000 bad loans. A leaderboard gap of that size is expected and does not indicate overfitting. A gap well above the simulated 95th percentile would point to a distribution problem rather than chance.

> Similar KS in the thin-file and history segments, and a flat month profile, show that the blend does not rely on a single sub-population or period. Those are the conditions for the public and private scores to stay close.

---

# Submission <a name="9"></a>

---

The blended test ranks are min-max scaled into [0, 1], which leaves the KS unchanged. The file is validated against every rule in the competition description before it is written.

In [ ]:
def to_unit(p):
    return (p - p.min()) / (p.max() - p.min())

def write_submission(pred, path):
    sub = pl.DataFrame({'application_id': test_ids, 'default_probability': to_unit(pred)})
    assert sub.height == sample_sub.height == T['test_applications'].height
    assert sub['application_id'].n_unique() == sub.height
    assert set(sub['application_id']) == set(sample_sub['application_id'])
    assert sub['default_probability'].is_finite().all()
    assert sub['default_probability'].is_between(0, 1).all()
    sub.write_csv(path)
    print('wrote', path, sub.shape)
    return sub

submission = write_submission(blend_te, CFG.OUTPUT_DIR / 'submission.csv')
gbdt = [i for i, n in enumerate(names) if n in ('lgb', 'xgb', 'cat')]
if gbdt:
    write_submission(R_TE[:, gbdt].mean(1), CFG.OUTPUT_DIR / 'submission_gbdt_only.csv')
print(submission.head())

#### Insights

> `submission.csv` is the final blend. `submission_gbdt_only.csv` is a fallback built only from the tree models, and it is useful as a second leaderboard data point. If the two differ on the public board by less than the simulated noise floor, choose the final submission by OOF KS.